In [1]:
#Import Julia packages
using XLSX 
using FastaIO 
using HTTP
using CodecZlib
using Revise
using CSV
using DataFrames
using Serialization

#### Including functions needed to run the notebook

In [2]:
function correct_path(splitted_path, String)
    """Function to retrieve files in a given path.
    It takes as input:
        1) splitted path: the current path split by each separator
        2) String: the path to reach the wanted file 
    The function returns:
        1) the correct complete path to reach the file
    """
    S="" #String initialization
    for i in 1:(length(splitted_path)-1)
        #Constructing the path with the correct separator and removing the last term
        S=S*splitted_path[i]*"/" 
    end
    #Final Output
    return S*String 
end

correct_path (generic function with 1 method)

In [3]:
#Julia file 
includet(correct_path(split(pwd(), '/'), "Functions/Mapping miCLIP functions.jl"));

#### Impote GRCh17 files 

> Transcripts annotation file

In [4]:
file_annotations_rna=HTTP.get("https://ftp.ncbi.nlm.nih.gov/refseq/H_sapiens/annotation/annotation_releases/105.20190906/GCF_000001405.25_GRCh37.p13/GCF_000001405.25_GRCh37.p13_rna.gbff.gz");
bytes=file_annotations_rna.body
io=GzipDecompressorStream(IOBuffer(bytes))
Annotations_rna=String(read(io));
gene_vec=Annotations_gene(Annotations_rna);

> Transcript sequences

In [5]:
file_transcripts=HTTP.get("https://ftp.ncbi.nlm.nih.gov/refseq/H_sapiens/annotation/annotation_releases/105.20190906/GCF_000001405.25_GRCh37.p13/GCF_000001405.25_GRCh37.p13_rna.fna.gz");
bytes=file_transcripts.body
decompressed=read(GzipDecompressorStream(IOBuffer(bytes)))
io=IOBuffer(decompressed)
reader=FastaReader(io);
human_transcripts=collect(reader);
long_transcripts,Dict_ind=longest_transcripts(human_transcripts,gene_vec);
indexes_long=sort(vcat(values(Dict_ind)...));

> Genomes annotations file

In [6]:
file_annotations_dna=HTTP.get("https://ftp.ncbi.nlm.nih.gov/refseq/H_sapiens/annotation/annotation_releases/105.20190906/GCF_000001405.25_GRCh37.p13/GCF_000001405.25_GRCh37.p13_genomic.gff.gz");
bytes=file_annotations_dna.body
io=GzipDecompressorStream(IOBuffer(bytes))
Annot=(String(read(io)));
Annotations_dna=Annotations_genome(Annot);

## miCLIP HEK293T: Train, Test

In [7]:
file_http=HTTP.get("https://static-content.springer.com/esm/art%3A10.1038%2Fnmeth.3453/MediaObjects/41592_2015_BFnmeth3453_MOESM77_ESM.xlsx");
io = IOBuffer(file_http.body )
Dataset_m6A= (XLSX.readxlsx(io)["Sheet1"][:])[2:end, :];

In [8]:
GENE_to_INFO_train_test=mapping_wrapped(human_transcripts,indexes_long,Annotations_dna,Annotations_rna,gene_vec,Dataset_m6A,long_transcripts);

In [9]:
#Training transcripts list used for GLORI 
genes_train=deserialize(pwd()*"/Data/gene_train.bin"); 
#Saving all training transcripts into a vocabulary
Dict_train=Dict()
[Dict_train[a]=[] for a in genes_train];
#Separating Train and Test miCLIP datasets, based on the transcript belonging to GLORI training dataset or not
Train, Test=Dict(), Dict()
[((haskey(Dict_train, a)==false)  ? Test[a]=GENE_to_INFO_train_test[a] : Train[a]=GENE_to_INFO_train_test[a]) for a in keys(GENE_to_INFO_train_test)];


### Prior Dataset

In [11]:
file_http=HTTP.get("https://static-content.springer.com/esm/art%3A10.1038%2Fnmeth.3453/MediaObjects/41592_2015_BFnmeth3453_MOESM78_ESM.xlsx");
io = IOBuffer(file_http.body )
Dataset_m6A_prior= (XLSX.readxlsx(io)["Sheet1"][:])[2:end, :];

In [12]:
GENE_to_INFO_prior=mapping_wrapped(human_transcripts,indexes_long,Annotations_dna,Annotations_rna,gene_vec,Dataset_m6A_prior,long_transcripts);

Saving miCLIP datasets

In [13]:
serialize(pwd()*"/Data/Train_miCLIP.bin", Train)
serialize(pwd()*"/Data/Test_miCLIP.bin", Test)
serialize(pwd()*"/Data/Prior_miCLIP.bin", GENE_to_INFO_prior)